# 08 · Synthesis: the three hypotheses, the robustness table and the report

Narrative only: the logic lives in `src/analysis/robustness.py` and `src/report/build_report.py`. Run `python run.py analyse figures report` first.

The report (`report/report.md`) is **generated** from `results/`: every number in it is a placeholder filled by the builder, and `results/report_values.json` maps each one to the file it came from. Re-running the builder after new model conditions have been embedded regenerates the whole paper.

In [ ]:
import json, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
import config
from IPython.display import Image, Markdown
load = lambda name: json.loads((config.RESULTS_DIR / name).read_text())

## 1. The three hypotheses on one panel

In [ ]:
Image(filename=str(config.FIGURES_DIR / 'fig11_synthesis.png'), width=1000)

In [ ]:
h1 = pd.read_csv(config.RESULTS_DIR / 'h1_alignment.csv')
h2 = pd.read_csv(config.RESULTS_DIR / 'h2_blindspot.csv')
h3 = pd.read_csv(config.RESULTS_DIR / 'h3_typology.csv')
print('H1 conditions done:', (h1['status'] == 'done').sum(), 'of', len(h1))
h1[['label','status','rho_raw','p_mantel_bh','h1_supported']]

In [ ]:
h2[(h2['design'] == 'matched') & (h2['primary'])][['label','n_present','n_absent','cliffs_delta','delta_ci_low','delta_ci_high','p_one_sided','audit_status']]

In [ ]:
h3[h3['confirmatory']][['label','proxy','auc_kind','status','rho','p_one_sided','h3_supported']]

## 2. Robustness: does each conclusion survive?
A check *survives* when it reproduces the primary conclusion (sign and significance), not when it reproduces the number.

In [ ]:
rs = pd.read_csv(config.RESULTS_DIR / 'robustness_summary.csv')
rs['conclusion'].value_counts()

In [ ]:
rs[['hypothesis','check','value_min','value_max','conclusion']]

In [ ]:
rob = pd.read_csv(config.RESULTS_DIR / 'robustness.csv')
rob[rob['check'].str.startswith('lofo_')][['check','value','p','verdict']].head(20)

## 3. Everything still pending

In [ ]:
rob[rob['verdict'] == 'PENDING USER RUN'][['hypothesis','check','description','note']]

## 4. The report and its provenance
Every value in `report/report.md`, with the results file it came from.

In [ ]:
rv = load('report_values.json')
print(rv['word_count_excluding_tables'], 'words outside tables;', rv['n_used'], 'of', rv['n_placeholders'], 'values used')
pd.DataFrame([{'key': k, 'value': v['value'][:40], 'source': v['source']} for k, v in rv['values'].items()]).head(40)

In [ ]:
Markdown((config.REPORT_DIR / 'report.md').read_text()[:4000])